In [ ]:
import pyomo.environ as pyo
import shutil

## If CBC is not installed, install it
if not shutil.which("cbc"):
  !apt-get install coinor-cbc
else:
  print("CBC is already installed")


# Create a Pyomo model
model = pyo.ConcreteModel()

# Define the sets
model.plants = pyo.Set(initialize=['Chandigarh', 'Raipur', 'Dharwad', 'Howrah'])
model.markets = pyo.Set(initialize=['Delhi', 'Mumbai', 'Bengaluru', 'Nagpur', 'Hyderabad', 'Chennai', 'Kolkatta', 'Patna', 'Lucknow', 'Ahmedabad'])

# Define the parameters
supply = {
    'Chandigarh': 500,
    'Raipur': 600,
    'Dharwad': 400,
    'Howrah': 700
}

demand = {
    'Delhi': 200,
    'Mumbai': 300,
    'Bengaluru': 250,
    'Nagpur': 150,
    'Hyderabad': 200,
    'Chennai': 200,
    'Kolkatta': 300,
    'Patna': 100,
    'Lucknow': 150,
    'Ahmedabad': 200
}

# Example transportation costs
costs = {
    ('Chandigarh', 'Delhi'): 10, ('Chandigarh', 'Mumbai'): 15, ('Chandigarh', 'Bengaluru'): 25, ('Chandigarh', 'Nagpur'): 20, ('Chandigarh', 'Hyderabad'): 22, ('Chandigarh', 'Chennai'): 28, ('Chandigarh', 'Kolkatta'): 30, ('Chandigarh', 'Patna'): 18, ('Chandigarh', 'Lucknow'): 12, ('Chandigarh', 'Ahmedabad'): 8,
    ('Raipur', 'Delhi'): 18, ('Raipur', 'Mumbai'): 12, ('Raipur', 'Bengaluru'): 20, ('Raipur', 'Nagpur'): 10, ('Raipur', 'Hyderabad'): 15, ('Raipur', 'Chennai'): 22, ('Raipur', 'Kolkatta'): 14, ('Raipur', 'Patna'): 16, ('Raipur', 'Lucknow'): 20, ('Raipur', 'Ahmedabad'): 16,
    ('Dharwad', 'Delhi'): 25, ('Dharwad', 'Mumbai'): 18, ('Dharwad', 'Bengaluru'): 10, ('Dharwad', 'Nagpur'): 15, ('Dharwad', 'Hyderabad'): 12, ('Dharwad', 'Chennai'): 18, ('Dharwad', 'Kolkatta'): 25, ('Dharwad', 'Patna'): 30, ('Dharwad', 'Lucknow'): 22, ('Dharwad', 'Ahmedabad'): 14,
    ('Howrah', 'Delhi'): 30, ('Howrah', 'Mumbai'): 25, ('Howrah', 'Bengaluru'): 28, ('Howrah', 'Nagpur'): 22, ('Howrah', 'Hyderabad'): 20, ('Howrah', 'Chennai'): 15, ('Howrah', 'Kolkatta'): 10, ('Howrah', 'Patna'): 12, ('Howrah', 'Lucknow'): 18, ('Howrah', 'Ahmedabad'): 25
}


model.supply = pyo.Param(model.plants, initialize=supply)
model.demand = pyo.Param(model.markets, initialize=demand)
model.costs = pyo.Param(model.plants, model.markets, initialize=costs)

# Define the decision variables
model.x = pyo.Var(model.plants, model.markets, within=pyo.NonNegativeReals)

# Define the objective function
def total_cost(model):
    return sum(model.costs[p, m] * model.x[p, m] for p in model.plants for m in model.markets)
model.objective = pyo.Objective(rule=total_cost, sense=pyo.minimize)

# Define a constraint function
def supply_constraint(model, p):
    return sum(model.x[p, m] for m in model.markets) <= model.supply[p]

# Put the above constraint function in the model
model.supply_constraint = pyo.Constraint(model.plants, rule=supply_constraint)

# define another constraint function
def demand_constraint(model, m):
    return sum(model.x[p, m] for p in model.plants) >= model.demand[m]

# Put these constraints also in the model
model.demand_constraint = pyo.Constraint(model.markets, rule=demand_constraint)

#now solve
solver = pyo.SolverFactory('cbc',executable='/usr/bin/cbc')
results = solver.solve(model)
# === Print results ===
print("\n=== Optimization Results ===")
print("Status:", results.solver.status)
print("Termination Condition:", results.solver.termination_condition)

# Print optimized cost
print("\nOptimized Total Cost =", pyo.value(model.objective))

# Print optimized allocations (only positive values)
print("\nOptimized Allocations:")
for p in model.plants:
    for m in model.markets:
        val = pyo.value(model.x[p, m])
        if val > 1e-6:  # avoid printing tiny numbers
            print(f"Send {val:.2f} units from {p} to {m}")


CBC is already installed

=== Optimization Results ===
Status: ok
Termination Condition: optimal

Optimized Total Cost = 23050.0

Optimized Allocations:
Send 200.00 units from Chandigarh to Delhi
Send 100.00 units from Chandigarh to Lucknow
Send 200.00 units from Chandigarh to Ahmedabad
Send 300.00 units from Raipur to Mumbai
Send 150.00 units from Raipur to Nagpur
Send 50.00 units from Raipur to Hyderabad
Send 250.00 units from Dharwad to Bengaluru
Send 150.00 units from Dharwad to Hyderabad
Send 200.00 units from Howrah to Chennai
Send 300.00 units from Howrah to Kolkatta
Send 100.00 units from Howrah to Patna
Send 50.00 units from Howrah to Lucknow


In [ ]:
import pyomo.environ as pyo

# Data
P = [155000, 120000, 100000, 70000, 60000, 100000]
L = [12000, 10000, 9000, 8000, 6000, 9000]
E = [350, 300, 100, 200, 100, 200]
n = len(P)

# Model
model = pyo.ConcreteModel()

# Variables
model.b0 = pyo.Var(domain=pyo.Reals)
model.b1 = pyo.Var(domain=pyo.Reals)
model.b2 = pyo.Var(domain=pyo.Reals)
model.R = pyo.Var(range(n), domain=pyo.NonNegativeReals)

# Constraints
def residual_upper(m, i):
    return P[i] - (m.b0 + m.b1*L[i] + m.b2*E[i]) <= m.R[i]
model.resid_up = pyo.Constraint(range(n), rule=residual_upper)

def residual_lower(m, i):
    return (m.b0 + m.b1*L[i] + m.b2*E[i]) - P[i] <= m.R[i]
model.resid_low = pyo.Constraint(range(n), rule=residual_lower)

# Objective (Minimize sum of absolute residuals)
model.obj = pyo.Objective(expr=sum(model.R[i] for i in range(n)), sense=pyo.minimize)

# Solver
solver = pyo.SolverFactory("cbc")
results = solver.solve(model, tee=False)

# Output
print("Optimal Cost (Sum of Absolute Residuals):", pyo.value(model.obj))
print("b0 =", pyo.value(model.b0))
print("b1 =", pyo.value(model.b1))
print("b2 =", pyo.value(model.b2))

print("\nWarehouse-wise Results:")
print("i   Actual_P     Predicted_P    Residual")
for i in range(n):
    Pi_hat = pyo.value(model.b0) + pyo.value(model.b1)*L[i] + pyo.value(model.b2)*E[i]
    Ri = P[i] - Pi_hat
    print(f"{i+1}   {P[i]:8.0f}   {Pi_hat:12.2f}   {Ri:9.2f}")


Optimal Cost (Sum of Absolute Residuals): 27142.8564
b0 = -55714.286
b1 = 17.142857
b2 = 14.285714

Warehouse-wise Results:
i   Actual_P     Predicted_P    Residual
1     155000      155000.00        0.00
2     120000      120000.00        0.00
3     100000      100000.00        0.00
4      70000       84285.71   -14285.71
5      60000       48571.43    11428.57
6     100000      101428.57    -1428.57


In [ ]:
from pyomo.environ import *

# --- Sets ---
days = ["Mon", "Tue", "Wed", "Thu", "Fri"]
candidates = ["K.C.", "D.H.", "H.B.", "S.C.", "K.S.", "N.K."]

# --- Parameters ---
wage = {
    "K.C.": 150, "D.H.": 152, "H.B.": 148,
    "S.C.": 146, "K.S.": 166, "N.K.": 176
}

availability = {
    ("K.C.", "Mon"): 6, ("K.C.", "Tue"): 0, ("K.C.", "Wed"): 6, ("K.C.", "Thu"): 0, ("K.C.", "Fri"): 6,
    ("D.H.", "Mon"): 0, ("D.H.", "Tue"): 6, ("D.H.", "Wed"): 0, ("D.H.", "Thu"): 6, ("D.H.", "Fri"): 0,
    ("H.B.", "Mon"): 4, ("H.B.", "Tue"): 8, ("H.B.", "Wed"): 4, ("H.B.", "Thu"): 0, ("H.B.", "Fri"): 4,
    ("S.C.", "Mon"): 5, ("S.C.", "Tue"): 5, ("S.C.", "Wed"): 5, ("S.C.", "Thu"): 0, ("S.C.", "Fri"): 5,
    ("K.S.", "Mon"): 3, ("K.S.", "Tue"): 0, ("K.S.", "Wed"): 3, ("K.S.", "Thu"): 8, ("K.S.", "Fri"): 0,
    ("N.K.", "Mon"): 0, ("N.K.", "Tue"): 0, ("N.K.", "Wed"): 0, ("N.K.", "Thu"): 6, ("N.K.", "Fri"): 2,
}

min_hours = {"K.C.": 8, "D.H.": 8, "H.B.": 8, "S.C.": 8, "K.S.": 7, "N.K.": 7}
H = {d: 14 for d in days}

# --- Model ---
model = ConcreteModel()
model.x = Var(candidates, days, domain=NonNegativeReals)

model.obj = Objective(expr=sum(wage[i] * model.x[i, d] for i in candidates for d in days),
                      sense=minimize)

model.cover = Constraint(days, rule=lambda m, d: sum(m.x[i, d] for i in candidates) == H[d])
model.minhrs = Constraint(candidates, rule=lambda m, i: sum(m.x[i, d] for d in days) >= min_hours[i])
model.avail = Constraint(candidates, days, rule=lambda m, i, d: m.x[i, d] <= availability[i, d])

# --- Solve with CBC ---
solver = SolverFactory("cbc")
solver.solve(model, tee=False)

# --- Output (answers only) ---
print("Optimal Schedule (hours per day):")
print(f"{'Day':<6} " + " ".join([f"{i:>6}" for i in candidates]))
for d in days:
    row = [f"{model.x[i, d]():6.1f}" for i in candidates]
    print(f"{d:<6} " + " ".join(row))

print("\nTotal Cost =", model.obj())


Optimal Schedule (hours per day):
Day      K.C.   D.H.   H.B.   S.C.   K.S.   N.K.
Mon       2.0    0.0    4.0    5.0    3.0    0.0
Tue       0.0    2.0    7.0    5.0    0.0    0.0
Wed       4.0    0.0    4.0    5.0    1.0    0.0
Thu       0.0    6.0    0.0    0.0    3.0    5.0
Fri       3.0    0.0    4.0    5.0    0.0    2.0

Total Cost = 10692.0


In [ ]:
import pyomo.environ as pyo

# -----------------
# Data
# -----------------
weeks = range(1, 9)
demand_swiss = {1:12,2:12,3:12,4:16,5:16,6:20,7:20,8:20}
demand_sharp = {1:8,2:8,3:10,4:10,5:12,6:12,7:12,8:12}

init_workers = 60
max_workers = 90
hours_per_week = 40
prod_swiss = 10 * hours_per_week   # 400 kg per worker per week
prod_sharp = 6 * hours_per_week    # 240 kg per worker per week

# -----------------
# Model
# -----------------
model = pyo.ConcreteModel()

# Decision variables
model.hire = pyo.Var(weeks, domain=pyo.NonNegativeIntegers)
model.x_swiss = pyo.Var(weeks, domain=pyo.NonNegativeReals)
model.x_sharp = pyo.Var(weeks, domain=pyo.NonNegativeReals)
model.inv_swiss = pyo.Var(weeks, domain=pyo.NonNegativeReals)
model.inv_sharp = pyo.Var(weeks, domain=pyo.NonNegativeReals)

# Workforce tracking
model.workers = pyo.Var(weeks, domain=pyo.NonNegativeReals)

# -----------------
# Constraints
# -----------------

# Workforce evolution
# trainers take 2 weeks → effective workers reduced
def workers_rule(m, t):
    if t == 1:
        return m.workers[t] == init_workers
    else:
        return m.workers[t] == m.workers[t-1] + m.hire[t-1]
model.workforce_balance = pyo.Constraint(weeks, rule=workers_rule)

# Production capacity (in 1000 kg)
def prod_capacity_rule(m, t):
    # Each worker can either produce swiss or sharp
    return m.x_swiss[t]/1000/prod_swiss + m.x_sharp[t]/1000/prod_sharp <= m.workers[t]
model.prod_capacity = pyo.Constraint(weeks, rule=prod_capacity_rule)

# Swiss inventory balance
model.inv_swiss_bal = pyo.ConstraintList()
for t in weeks:
    if t == 1:
        model.inv_swiss_bal.add(model.x_swiss[t] - demand_swiss[t] == model.inv_swiss[t])
    else:
        model.inv_swiss_bal.add(model.inv_swiss[t-1] + model.x_swiss[t] - demand_swiss[t] == model.inv_swiss[t])

# Sharp inventory balance
model.inv_sharp_bal = pyo.ConstraintList()
for t in weeks:
    if t == 1:
        model.inv_sharp_bal.add(model.x_sharp[t] - demand_sharp[t] == model.inv_sharp[t])
    else:
        model.inv_sharp_bal.add(model.inv_sharp[t-1] + model.x_sharp[t] - demand_sharp[t] == model.inv_sharp[t])

# Inventory cannot be kept >1 week
def inv_limit_rule(m, t):
    return m.inv_swiss[t] <= demand_swiss.get(t+1, 999)
model.inv_limit = pyo.Constraint(weeks, rule=inv_limit_rule)

# Workforce cap
def max_workers_rule(m, t):
    return m.workers[t] <= max_workers
model.max_workers_con = pyo.Constraint(weeks, rule=max_workers_rule)

# -----------------
# Objective: Minimize labor cost
# -----------------
cost_per_worker = 1  # normalize to 1 per worker per week
model.obj = pyo.Objective(expr=sum(model.workers[t] * cost_per_worker for t in weeks), sense=pyo.minimize)

# -----------------
# Solve
# -----------------
solver = pyo.SolverFactory('cbc')
results = solver.solve(model, tee=False)

# -----------------
# Results
# -----------------
print("Week | Hires | Workers | SwissProd | SharpProd | InvSwiss | InvSharp")
for t in weeks:
    print(t,
          pyo.value(model.hire[t]),
          pyo.value(model.workers[t]),
          pyo.value(model.x_swiss[t]),
          pyo.value(model.x_sharp[t]),
          pyo.value(model.inv_swiss[t]),
          pyo.value(model.inv_sharp[t]))


ERROR:pyomo.common.numeric_types:evaluating object as numeric value: hire[8]
    (object: <class 'pyomo.core.base.var.VarData'>)
No value for uninitialized VarData object hire[8]


Week | Hires | Workers | SwissProd | SharpProd | InvSwiss | InvSharp
1 0.0 60.0 12.0 16.0 0.0 8.0
2 0.0 60.0 12.0 0.0 0.0 0.0
3 0.0 60.0 12.0 10.0 0.0 0.0
4 0.0 60.0 16.0 10.0 0.0 0.0
5 0.0 60.0 16.0 12.0 0.0 0.0
6 0.0 60.0 20.0 12.0 0.0 0.0
7 0.0 60.0 20.0 12.0 0.0 0.0


ValueError: No value for uninitialized VarData object hire[8]